# Credit Classifier Comparison

This notebook compares logistic regression, Naive Bayes, random forest, and SVM. Cross-validation tunes the SVM on the training data, while the test set is reserved for the final comparison.

> Pickle files may execute code when loaded. This notebook accepts only the artifact tracked in this private repository; do not replace it with an unverified download.

In [1]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [2]:
candidates = [
    Path('data/credit.pkl'),
    Path('credit-classifiers/data/credit.pkl'),
]
legacy_path = next((path for path in candidates if path.is_file()), None)
if legacy_path is None:
    raise FileNotFoundError('credit.pkl was not found')
with legacy_path.open('rb') as file:
    X_train, y_train, X_test, y_test = pickle.load(file)[:4]
len(X_train), len(X_test)

(1500, 500)

In [3]:
models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(random_state=42)),
    ]),
    'naive_bayes': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB()),
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42
    ),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, predictions),
        'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
    })

In [4]:
svm = GridSearchCV(
    Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
    {'model__kernel': ['linear', 'rbf', 'poly'], 'model__C': [0.1, 1, 10]},
    scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1,
)
svm.fit(X_train, y_train)
predictions = svm.predict(X_test)
rows.append({
    'model': 'svm',
    'accuracy': accuracy_score(y_test, predictions),
    'macro_f1': f1_score(y_test, predictions, average='macro', zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, predictions),
})
results = pd.DataFrame(rows).set_index('model').sort_values('macro_f1', ascending=False)
print('SVM selected on the training data:', svm.best_params_)
results

SVM selected on the training data: {'model__C': 10, 'model__kernel': 'rbf'}


,accuracy,macro_f1,balanced_accuracy
model,,,
svm,0.988,0.973475,0.979788
random_forest,0.986,0.968000,0.958644
logistic_regression,0.946,0.878237,0.875717
naive_bayes,0.938,0.845357,0.811138
